# 04b — pySCENIC Cross-Dataset Filter, v2 (power-matched)

**Why this notebook exists:** an independent audit of the Stage 3a pipeline (before Stage 7 feature selection) flagged that the original cross-dataset "consistent regulon" filter (`04_pyscenic_analysis.ipynb`, cell 6) has two problems:

1. It selects by `head(50)` (rank) + sign agreement, which silently drops the BH-corrected p-values computed one cell earlier — not a formally corrected joint test.
2. TSPC has far fewer cells than T-FAP in **both** datasets (Harvey: TSPC n=266 vs T-FAP n=433; Cherief: TSPC n=451 vs T-FAP n=1245). Wilcoxon z-scores scale with n for a fixed effect size, so a rank-based cutoff ("top 50") structurally favours the larger group. This raised the concrete question: **is "0 consistent TSPC regulons" real biology, or an artifact of TSPC being underpowered in both datasets relative to T-FAP?**

**This notebook does not replace `04_pyscenic_analysis.ipynb` — that stays as the original record.** This is a new, independent re-analysis of the same AUCell scores, using a power-matched, stability-selection-style test, so the two can be compared side by side.

**Method:** for each dataset, repeatedly (200 iterations) downsample the majority group (T-FAP) to the minority group's cell count (TSPC), rerun the same Wilcoxon test (BH-corrected within each iteration) on the size-matched groups, and report each regulon's **selection frequency** — the fraction of the 200 iterations in which it was significant (`pvals_adj < 0.05`). A regulon is called "power-matched significant" if its selection frequency is ≥ 0.95 in a dataset. The cross-dataset "consistent" list requires ≥ 0.95 selection frequency in **both** datasets with the same sign — the same logical structure as the original filter, but now free of the sample-size asymmetry and using an aggregate over repeated corrected tests rather than a single rank cutoff.

In [1]:
import h5py
import loompy
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path

sc.settings.verbosity = 1

ROOT = Path('../..')
DATA = ROOT / 'data' / 'pyscenic'


def load_aucell_loom(aucell_path, input_loom_path=None):
    """Same loader as 04_pyscenic_analysis.ipynb -- unchanged."""
    with h5py.File(str(aucell_path), 'r') as f:
        col_keys = list(f['col_attrs'].keys())
        reg_auc = f['col_attrs']['RegulonsAUC']
        regulon_names = list(reg_auc.dtype.names)
        auc = np.column_stack([reg_auc[r][:] for r in regulon_names])
        cell_key = 'CellID' if 'CellID' in col_keys else 'obs_names'
        cells = f['col_attrs'][cell_key][:].astype(str)
        cell_type = f['col_attrs']['cell_type'][:].astype(str) if 'cell_type' in col_keys else None
    adata = sc.AnnData(X=auc.astype(np.float32), obs=pd.DataFrame(index=cells), var=pd.DataFrame(index=regulon_names))
    if cell_type is not None:
        adata.obs['cell_type'] = cell_type
    elif input_loom_path is not None:
        with loompy.connect(str(input_loom_path), 'r') as ds:
            ct_map = pd.Series(ds.ca['cell_type'].astype(str), index=ds.ca['obs_names'].astype(str))
        adata.obs['cell_type'] = adata.obs.index.map(ct_map)
    adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')
    return adata


adata_h = load_aucell_loom(DATA / 'harvey_aucell.loom', DATA / 'harvey_tspc_tfap.loom')
adata_c = load_aucell_loom(DATA / 'cherief_aucell.loom', DATA / 'cherief_tspc_tfap.loom')

print('Harvey  :', adata_h.obs['cell_type'].value_counts().to_dict())
print('Cherief :', adata_c.obs['cell_type'].value_counts().to_dict())

Harvey  : {'T-FAP': 433, 'TSPC': 266}
Cherief : {'T-FAP': 1245, 'TSPC': 451}


## 1. Power-matched differential regulon test (stability selection over repeated downsampling)

In [2]:
def power_matched_diff_regulons(adata, label, n_iter=200, seed0=0):
    counts = adata.obs['cell_type'].value_counts()
    minority_n = counts.min()
    majority_group = counts.idxmax()
    minority_group = counts.idxmin()
    print(f'{label}: minority={minority_group} n={minority_n}, downsampling {majority_group} '
          f'({counts.max()}) to n={minority_n} across {n_iter} iterations')

    regulons = adata.var_names
    sel_freq = {g: pd.Series(0, index=regulons) for g in ['TSPC', 'T-FAP']}
    score_sum = {g: pd.Series(0.0, index=regulons) for g in ['TSPC', 'T-FAP']}

    minority_idx = adata.obs_names[adata.obs['cell_type'] == minority_group]
    majority_idx_all = adata.obs_names[adata.obs['cell_type'] == majority_group]

    rng = np.random.default_rng(seed0)
    for i in range(n_iter):
        majority_sample = rng.choice(majority_idx_all, size=minority_n, replace=False)
        keep = list(minority_idx) + list(majority_sample)
        sub = adata[keep].copy()
        sc.tl.rank_genes_groups(sub, groupby='cell_type', groups=['TSPC', 'T-FAP'],
                                 reference='rest', method='wilcoxon', use_raw=False)
        for group in ['TSPC', 'T-FAP']:
            df = sc.get.rank_genes_groups_df(sub, group=group).set_index('names')
            sig = df['pvals_adj'] < 0.05
            sel_freq[group] += sig.reindex(regulons).fillna(False).astype(int)
            score_sum[group] += df['scores'].reindex(regulons).fillna(0.0)

    results = {}
    for group in ['TSPC', 'T-FAP']:
        out = pd.DataFrame({
            'names': regulons,
            'selection_freq': sel_freq[group].values / n_iter,
            'mean_score': score_sum[group].values / n_iter,
        }).sort_values('selection_freq', ascending=False).reset_index(drop=True)
        results[group] = out
        n_sig = (out['selection_freq'] >= 0.95).sum()
        print(f'{label} {group}: {n_sig} regulons with selection_freq >= 0.95 (of {len(out)} tested)')
    return results


diff_h_v2 = power_matched_diff_regulons(adata_h, 'Harvey', n_iter=200, seed0=1)
diff_c_v2 = power_matched_diff_regulons(adata_c, 'Cherief', n_iter=200, seed0=2)

Harvey: minority=TSPC n=266, downsampling T-FAP (433) to n=266 across 200 iterations


Harvey TSPC: 68 regulons with selection_freq >= 0.95 (of 153 tested)
Harvey T-FAP: 68 regulons with selection_freq >= 0.95 (of 153 tested)
Cherief: minority=TSPC n=451, downsampling T-FAP (1245) to n=451 across 200 iterations


Cherief TSPC: 63 regulons with selection_freq >= 0.95 (of 157 tested)
Cherief T-FAP: 63 regulons with selection_freq >= 0.95 (of 157 tested)


## 2. Cross-dataset consistent regulons (power-matched)

Same logical structure as the original filter (significant in both datasets, same direction), but using the power-matched selection frequency instead of a raw rank cutoff.

In [3]:
THRESH = 0.95
shared_v2 = {}
for ct in ['TSPC', 'T-FAP']:
    h_df = diff_h_v2[ct].set_index('names')
    c_df = diff_c_v2[ct].set_index('names')
    h_sig = set(h_df[h_df['selection_freq'] >= THRESH].index)
    c_sig = set(c_df[c_df['selection_freq'] >= THRESH].index)
    common = h_sig & c_sig
    consistent = [r for r in common if h_df.loc[r, 'mean_score'] > 0 and c_df.loc[r, 'mean_score'] > 0]
    shared_v2[ct] = sorted(consistent)
    print(f'{ct}: {len(shared_v2[ct])} power-matched consistent regulons')

rows = []
for ct, regs in shared_v2.items():
    h_df = diff_h_v2[ct].set_index('names')
    c_df = diff_c_v2[ct].set_index('names')
    for reg in regs:
        rows.append({
            'regulon': reg, 'cell_type': ct,
            'harvey_selection_freq': h_df.loc[reg, 'selection_freq'],
            'harvey_mean_score': h_df.loc[reg, 'mean_score'],
            'cherief_selection_freq': c_df.loc[reg, 'selection_freq'],
            'cherief_mean_score': c_df.loc[reg, 'mean_score'],
        })
shared_v2_df = pd.DataFrame(rows).sort_values(['cell_type', 'harvey_mean_score'], ascending=[True, False])
shared_v2_df

TSPC: 0 power-matched consistent regulons
T-FAP: 15 power-matched consistent regulons


,regulon,cell_type,harvey_selection_freq,harvey_mean_score,cherief_selection_freq,cherief_mean_score
1,Cebpd(+),T-FAP,1.000,18.670223,1.00,24.179847
2,Egr1(+),T-FAP,1.000,18.525121,1.00,21.365297
5,Irf1(+),T-FAP,1.000,18.459903,1.00,17.205113
8,Jund(+),T-FAP,1.000,16.334259,1.00,23.490115
4,Fosb(+),T-FAP,1.000,16.041352,1.00,22.053967
7,Junb(+),T-FAP,1.000,15.603481,1.00,22.520071
6,Jun(+),T-FAP,1.000,15.105382,1.00,21.717749
11,Nfkb1(+),T-FAP,1.000,13.567824,1.00,17.529770
3,Fos(+),T-FAP,1.000,12.814728,1.00,14.476424
0,Atf3(+),T-FAP,1.000,9.898054,1.00,21.218248


## 3. Compare to the original (v1) result

Did power-matching change the headline claim?

In [4]:
v1 = pd.read_csv(DATA / 'shared_diff_regulons.csv')
v1_tf = set(v1[v1['cell_type'] == 'T-FAP']['regulon'].str.replace(r'\(\+\)$', '', regex=True))
v1_tspc = set(v1[v1['cell_type'] == 'TSPC']['regulon'].str.replace(r'\(\+\)$', '', regex=True))
v2_tf = set(r.replace('(+)', '') for r in shared_v2['T-FAP'])
v2_tspc = set(r.replace('(+)', '') for r in shared_v2['TSPC'])

print(f'v1: T-FAP n={len(v1_tf)}, TSPC n={len(v1_tspc)}')
print(f'v2: T-FAP n={len(v2_tf)}, TSPC n={len(v2_tspc)}')
print()
print('T-FAP dropped in v2 (was in v1, not in v2):', sorted(v1_tf - v2_tf))
print('T-FAP new in v2 (not in v1)              :', sorted(v2_tf - v1_tf))
print('T-FAP in both v1 and v2                  :', sorted(v1_tf & v2_tf))
print()
print('TSPC in v2:', sorted(v2_tspc) if v2_tspc else '(still empty)')

v1: T-FAP n=15, TSPC n=0
v2: T-FAP n=15, TSPC n=0

T-FAP dropped in v2 (was in v1, not in v2): ['Prdm16']
T-FAP new in v2 (not in v1)              : ['Zfx']
T-FAP in both v1 and v2                  : ['Atf3', 'Cebpd', 'Egr1', 'Fos', 'Fosb', 'Irf1', 'Jun', 'Junb', 'Jund', 'Klf6', 'Klf9', 'Nfkb1', 'Pbx1', 'Zfp369']

TSPC in v2: (still empty)


**Result: the T-FAP list is essentially unchanged (14 of 15 regulons replicate; Prdm16 drops out, Zfx enters, both borderline), and TSPC still returns zero consistent regulons — even after equalizing statistical power between the two cell types.**

This is a real test of the audit's hypothesis, and the hypothesis did **not** hold up: power asymmetry between TSPC (n=266/451) and T-FAP (n=433/1245) is not what's producing the "0 TSPC regulons" result. That strengthens confidence that the original Stage 3a finding reflects real biology (TSPC identity is more context/state-dependent than a set of consistently active TFs can capture, as discussed in the intermediate reflection), not a sample-size artifact.

## 4. Where do the audit's specific TSPC candidates (Cux1, Klf3, Mxd4) actually fail?

The audit noted these three regulons look like plausible Cherief TSPC hits that just don't clear Harvey's original top-50 rank cutoff. Checking them directly in the power-matched Harvey result shows *why* they fail — and it isn't a power problem.

In [5]:
h_tspc = diff_h_v2['TSPC'].set_index('names')
c_tspc = diff_c_v2['TSPC'].set_index('names')
for reg in ['Cux1(+)', 'Klf3(+)', 'Mxd4(+)']:
    h_row = h_tspc.loc[reg] if reg in h_tspc.index else None
    c_row = c_tspc.loc[reg] if reg in c_tspc.index else None
    print(reg)
    print('  Harvey :', 'not detected as a regulon in Harvey (TF/motif not called by cisTarget)' if h_row is None
          else f"selection_freq={h_row['selection_freq']:.2f}, mean_score={h_row['mean_score']:.2f}")
    print('  Cherief:', None if c_row is None else f"selection_freq={c_row['selection_freq']:.2f}, mean_score={c_row['mean_score']:.2f}")

Cux1(+)
  Harvey : not detected as a regulon in Harvey (TF/motif not called by cisTarget)
  Cherief: selection_freq=1.00, mean_score=23.09
Klf3(+)
  Harvey : selection_freq=0.05, mean_score=1.56
  Cherief: selection_freq=1.00, mean_score=20.53
Mxd4(+)
  Harvey : not detected as a regulon in Harvey (TF/motif not called by cisTarget)
  Cherief: selection_freq=1.00, mean_score=19.11


Cux1 and Mxd4 aren't even called as regulons in Harvey at all (cisTarget didn't detect a motif-supported target set for them there) — that's a dataset-level absence, not a power problem. Klf3 *is* present in Harvey but has selection_freq = 0.05 and mean_score ≈ 1.6 — genuinely null in Harvey even at matched sample size, not underpowered. So the specific mechanism the audit proposed (rank cutoff hiding real-but-underpowered TSPC regulons) doesn't explain these cases either.

## 5. Save power-matched outputs (separate from v1 — both are kept for the record)

In [6]:
for name, diff in [('harvey', diff_h_v2), ('cherief', diff_c_v2)]:
    for ct, df in diff.items():
        out = DATA / f'{name}_diff_regulons_v2_powermatched_{ct}.csv'
        df.to_csv(out, index=False)
        print('Saved:', out)

out = DATA / 'shared_diff_regulons_v2_powermatched.csv'
shared_v2_df.to_csv(out, index=False)
print('Saved:', out)

Saved: ..\..\data\pyscenic\harvey_diff_regulons_v2_powermatched_TSPC.csv
Saved: ..\..\data\pyscenic\harvey_diff_regulons_v2_powermatched_T-FAP.csv
Saved: ..\..\data\pyscenic\cherief_diff_regulons_v2_powermatched_TSPC.csv
Saved: ..\..\data\pyscenic\cherief_diff_regulons_v2_powermatched_T-FAP.csv
Saved: ..\..\data\pyscenic\shared_diff_regulons_v2_powermatched.csv


## 6. Conclusion for Stage 6/7

The Stage 3a "15 consistent T-FAP regulons / 0 consistent TSPC regulons" result **replicates** under a power-matched, stability-selection re-analysis: 14/15 T-FAP regulons hold (Zfx replaces the borderline Prdm16), and TSPC remains at zero. The candidate feature-set asymmetry already built into Stage 6/7 (TF regulon targets for T-FAP, expression markers only for TSPC) is not an artifact of the original ad hoc filter — it's a stable result across two different statistical approaches.

**What this does *not* rule out:** both datasets' *absolute* TSPC sample sizes (266, 451) are still small in absolute terms — power-matching equalizes TSPC vs. T-FAP *within* each dataset, but doesn't increase the total amount of information available about TSPC. A true small-but-real cross-dataset-consistent TSPC TF program could still be missed if its effect size is modest. This is a limitation of the source data (no more Harvey/Cherief TSPC cells exist to add), not something further re-analysis can fix — worth stating as a caveat rather than a re-open-the-question item.